In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install ultralytics

In [ ]:
import yaml
import os

# 1. Đường dẫn thư mục gốc dataset của bạn
dataset_root = '/kaggle/input/datasets/namproovipp/4-class-trash-detect-high-quality-dataset/dataset'

# Đường dẫn tới file classes.txt
classes_file = os.path.join(dataset_root, 'classes.txt')

# 2. Đọc danh sách các class từ file classes.txt
with open(classes_file, 'r') as f:
    # Đọc từng dòng, xóa khoảng trắng thừa và bỏ qua dòng trống
    class_names = [line.strip() for line in f.readlines() if line.strip()]

# 3. Tạo nội dung cho file data.yaml
data = {
    'path': dataset_root,  # Thư mục gốc chứa dữ liệu
    
    # CHÚ Ý: Nếu trong thư mục 'images' của bạn CÓ chia sẵn thư mục con 'train' và 'val' thì dùng 2 dòng này:
    # 'train': 'images/train',
    # 'val': 'images/val',
    
    # NẾU KHÔNG CHIA (tất cả ảnh vứt chung trong thư mục 'images'), thì cấu hình như sau để YOLO tự lấy ảnh:
    'train': 'images',
    'val': 'images',  
    
    'nc': len(class_names),  # Số lượng class (đếm tự động)
    'names': class_names     # Danh sách tên class
}

# 4. Lưu thành file data_kaggle.yaml vào thư mục có quyền ghi
yaml_path = '/kaggle/working/data_kaggle.yaml'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, sort_keys=False)

print(f"✅ Đã tạo file YAML thành công tại: {yaml_path}")
print("-" * 30)
print("NỘI DUNG FILE YAML:")
with open(yaml_path, 'r') as f:
    print(f.read())

In [ ]:
import os
import random
import shutil
import yaml

# 1. Đường dẫn gốc (Chỉ đọc)
input_dir = '/kaggle/input/datasets/namproovipp/4-class-trash-detect-high-quality-dataset/dataset'
images_input = os.path.join(input_dir, 'images')
labels_input = os.path.join(input_dir, 'labels')

# 2. Đường dẫn mới trên Kaggle (Được phép ghi)
working_dir = '/kaggle/working/dataset'

# Tạo sẵn các thư mục con train/val
for split in ['train', 'val']:
    os.makedirs(os.path.join(working_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(working_dir, 'labels', split), exist_ok=True)

# 3. Lấy danh sách ảnh và xáo trộn ngẫu nhiên
images = [f for f in os.listdir(images_input) if f.endswith(('.jpg', '.png', '.jpeg'))]
random.shuffle(images)

# Chia tỷ lệ 80% cho Train, 20% cho Validation
split_idx = int(len(images) * 0.8)
train_images = images[:split_idx]
val_images = images[split_idx:]

# Hàm copy ảnh và label tương ứng
def split_data(image_list, split_name):
    for img_name in image_list:
        # Copy ảnh
        shutil.copy(os.path.join(images_input, img_name), 
                    os.path.join(working_dir, 'images', split_name, img_name))
        
        # Tìm và copy file txt label tương ứng
        label_name = os.path.splitext(img_name)[0] + '.txt'
        label_path = os.path.join(labels_input, label_name)
        if os.path.exists(label_path):
            shutil.copy(label_path, 
                        os.path.join(working_dir, 'labels', split_name, label_name))

print(f"Đang tiến hành chia {len(images)} ảnh...")
split_data(train_images, 'train')
split_data(val_images, 'val')
print("✅ Đã chia xong dữ liệu vào /kaggle/working/dataset/")

# 4. Tự động đọc file classes.txt và tạo data.yaml
classes_file = os.path.join(input_dir, 'classes.txt')
with open(classes_file, 'r') as f:
    class_names = [line.strip() for line in f.readlines() if line.strip()]

yaml_data = {
    'path': working_dir,          # Trỏ về thư mục working mới
    'train': 'images/train',      # Thư mục train ta vừa copy
    'val': 'images/val',          # Thư mục val ta vừa copy
    'nc': len(class_names),
    'names': class_names
}

yaml_path = '/kaggle/working/data_kaggle.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_data, f, sort_keys=False)

print(f"✅ Đã tạo file YAML tại: {yaml_path}")


In [ ]:
!yolo detect train model=yolo11n.pt data=/kaggle/working/data_kaggle.yaml epochs=50 imgsz=640 batch=60 device='0,1' workers=2 cache=true

In [ ]:
import os
from IPython.display import FileLink

# 1. Nén toàn bộ thư mục train3 thành 1 file zip tên là "ket_qua_yolo.zip"
# Dấu chấm than (!) dùng để chạy lệnh bash trực tiếp trong ô code Python
!zip -r /kaggle/working/ket_qua_yolo.zip /kaggle/working/runs/detect/train

# 2. Tạo đường link để bạn click vào tải về
print("🎉 Đã nén xong! Bấm vào đường link bên dưới để tải về máy:")
FileLink('ket_qua_yolo.zip')